# Codificacion Variables Categorigas, Angulares y Temporales con metodo por Transecto y metodo General

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 3 (adaptado a transectos): Codificación de variables.

Entrada:
    - imputed_by_transect/
    - imputed_global/

Salida:
    - encoded/ml/by_transect/
    - encoded/dl/by_transect/
    - encoded/ml/global/
    - encoded/dl/global/
"""

import os
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

BASE_DIR = os.path.expanduser("/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/")
INPUT_BY_TRANSECT = os.path.join(BASE_DIR, "imputed_by_transect")
INPUT_GLOBAL = os.path.join(BASE_DIR, "imputed_global")

OUTPUT_BASE = os.path.join(BASE_DIR, "encoded")
OUTPUT_ML_TRANSECT = os.path.join(OUTPUT_BASE, "ml", "by_transect")
OUTPUT_DL_TRANSECT = os.path.join(OUTPUT_BASE, "dl", "by_transect")
OUTPUT_ML_GLOBAL = os.path.join(OUTPUT_BASE, "ml", "global")
OUTPUT_DL_GLOBAL = os.path.join(OUTPUT_BASE, "dl", "global")

for path in [OUTPUT_ML_TRANSECT, OUTPUT_DL_TRANSECT, OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL]:
    os.makedirs(path, exist_ok=True)

NUM_COLS = ['NO', 'NO2', 'NOx', 'O3', 'Veloc.', 'Direc.', 'Temp.', 'R.Sol.', 'Dist.', 'Angulo']
CATEGORICAL_COLS = ['Estacion', 'Transecto']

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def cyclical_encode(series, period):
    """Convierte una serie numérica a seno y coseno con el período dado."""
    rad = 2 * np.pi * series / period
    return np.sin(rad), np.cos(rad)


def standardize_target_column(df):
    """
    Garantiza que la variable objetivo se llame O3.
    Si existe O3_for_impute, la usa como respaldo para O3 y luego elimina O3_for_impute.
    """
    df = df.copy()

    if "O3_for_impute" in df.columns and "O3" not in df.columns:
        df.rename(columns={"O3_for_impute": "O3"}, inplace=True)

    elif "O3_for_impute" in df.columns and "O3" in df.columns:
        df["O3"] = df["O3"].where(df["O3"].notna(), df["O3_for_impute"])
        df.drop(columns=["O3_for_impute"], inplace=True)

    return df


def add_datetime_features(df):
    """
    A partir del índice datetime extrae:
    hora, día del año, semana y mes en forma cíclica.
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("El índice debe ser DatetimeIndex")

    idx = df.index
    hour = idx.hour
    dayofyear = idx.dayofyear
    week = idx.isocalendar().week.astype(int)
    month = idx.month
    year = idx.year

    df['hour_sin'], df['hour_cos'] = cyclical_encode(hour, 24)
    df['day_sin'], df['day_cos'] = cyclical_encode(dayofyear, 365)
    df['week_sin'], df['week_cos'] = cyclical_encode(week, 52)
    df['month_sin'], df['month_cos'] = cyclical_encode(month, 12)
    df['year'] = year
    return df


def add_directional_features(df):
    """
    Convierte Direc. y Angulo a seno y coseno.
    Las columnas originales se eliminan.
    """
    df = df.copy()

    for col in ['Direc.', 'Angulo']:
        if col in df.columns:
            values = pd.to_numeric(df[col], errors="coerce")
            rad = np.radians(values)
            df[f'{col}sin'] = np.sin(rad)
            df[f'{col}cos'] = np.cos(rad)
            df.drop(columns=[col], inplace=True)

    return df


def normalize_station(val):
    """
    Convierte un valor de estación a formato Estacion_N.
    """
    s = str(val).strip()

    m = re.search(r'Estacion[ _]?(\d+)', s, re.IGNORECASE)
    if m:
        return f"Estacion_{m.group(1)}"

    m = re.search(r'\bE[ _]?(\d+)\b', s, re.IGNORECASE)
    if m:
        return f"Estacion_{m.group(1)}"

    m = re.search(r'\d+', s)
    if m:
        return f"Estacion_{m.group(0)}"

    return s.replace(' ', '_')


def normalize_transect(val):
    """
    Convierte un valor de transecto a formato Transecto_N.
    """
    s = str(val).strip()

    m = re.search(r'Transecto[ _]?(\d+)', s, re.IGNORECASE)
    if m:
        return f"Transecto_{m.group(1)}"

    m = re.search(r'\bT[ _]?(\d+)\b', s, re.IGNORECASE)
    if m:
        return f"Transecto_{m.group(1)}"

    m = re.search(r'\d+', s)
    if m:
        return f"Transecto_{m.group(0)}"

    return s.replace(' ', '_')


def clean_column_names(col_names):
    """Limpia nombres de columnas por si aparece algún carácter problemático."""
    cleaned = []
    for name in col_names:
        name = str(name)
        name = name.replace(' ', '_').replace('/', '_')
        cleaned.append(name)
    return cleaned


def encode_categorical_ml(df, cat_cols):
    """
    Aplica one hot encoding a las columnas categóricas, previa normalización.
    """
    df = df.copy()

    existing = [c for c in cat_cols if c in df.columns]
    if not existing:
        return df

    if 'Estacion' in existing:
        df['Estacion'] = df['Estacion'].apply(normalize_station)
    if 'Transecto' in existing:
        df['Transecto'] = df['Transecto'].apply(normalize_transect)

    dummies = pd.get_dummies(df[existing].astype(str), prefix='', prefix_sep='')
    dummies.columns = clean_column_names(dummies.columns)

    df = pd.concat([df, dummies], axis=1)
    df.drop(columns=existing, inplace=True)

    return df


def encode_categorical_dl(df, cat_cols, save_mapping=True, mapping_file=None):
    """
    Convierte columnas categóricas a enteros usando factorize.
    Guarda el mapeo categoría a código si se solicita.
    """
    df = df.copy()

    existing = [c for c in cat_cols if c in df.columns]
    if not existing:
        return df, {}

    mapping = {}

    for col in existing:
        codes, uniques = pd.factorize(df[col], sort=False)
        df[col] = codes
        mapping[col] = {str(category): int(code) for code, category in enumerate(uniques)}

    if save_mapping and mapping_file:
        with open(mapping_file, 'w', encoding='utf-8') as f:
            json.dump(mapping, f, indent=2, ensure_ascii=False)

    return df, mapping


def prepare_dataframe(df, station_name=None, transect_clean=None):
    """
    Prepara el DataFrame para la codificación.
    """
    df = df.copy()

    df = standardize_target_column(df)

    if "Estacion" not in df.columns and station_name is not None:
        df["Estacion"] = station_name
    elif "Estacion" in df.columns and station_name is not None:
        df["Estacion"] = df["Estacion"].fillna(station_name)

    if "Transecto" in df.columns and transect_clean is not None:
        df["Transecto"] = df["Transecto"].fillna(transect_clean.replace("_", " "))

    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, errors="coerce")

    df = df.loc[~df.index.isna()].copy()
    df = df.sort_index(kind="mergesort")

    return df


def process_file(input_path, output_ml_dir, output_dl_dir, is_global=False):
    """
    Procesa un archivo CSV y guarda versiones para ML y DL.
    """
    base_name = Path(input_path).stem
    print(f"Procesando: {base_name} (global={is_global})")

    df = pd.read_csv(input_path, index_col=0, parse_dates=True, low_memory=False)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, errors="coerce")

    df = prepare_dataframe(df)

    df = add_datetime_features(df)
    df = add_directional_features(df)

    df_ml = df.copy()
    df_dl = df.copy()

    df_ml = encode_categorical_ml(df_ml, CATEGORICAL_COLS)

    mapping_file = os.path.join(output_dl_dir, f"{base_name}_mapping.json")
    df_dl, _ = encode_categorical_dl(
        df_dl,
        CATEGORICAL_COLS,
        save_mapping=True,
        mapping_file=mapping_file
    )

    ml_path = os.path.join(output_ml_dir, f"{base_name}.csv")
    dl_path = os.path.join(output_dl_dir, f"{base_name}.csv")

    df_ml.to_csv(ml_path, index=True)
    df_dl.to_csv(dl_path, index=True)

    print(f"  ML guardado en {ml_path}")
    print(f"  DL guardado en {dl_path}")


# ============================================================================
# PROCESAMIENTO
# ============================================================================

def process_by_transect():
    """Procesa todos los archivos de la carpeta imputed_by_transect."""
    if not os.path.exists(INPUT_BY_TRANSECT):
        print(f"La carpeta {INPUT_BY_TRANSECT} no existe. Se omite.")
        return

    files = list(Path(INPUT_BY_TRANSECT).glob("*.csv"))
    if not files:
        print(f"No se encontraron archivos CSV en {INPUT_BY_TRANSECT}")
        return

    for f in files:
        process_file(f, OUTPUT_ML_TRANSECT, OUTPUT_DL_TRANSECT, is_global=False)


def process_global():
    """Procesa todos los archivos de la carpeta imputed_global."""
    if not os.path.exists(INPUT_GLOBAL):
        print(f"La carpeta {INPUT_GLOBAL} no existe. Se omite.")
        return

    files = list(Path(INPUT_GLOBAL).glob("*.csv"))
    if not files:
        print(f"No se encontraron archivos CSV en {INPUT_GLOBAL}")
        return

    for f in files:
        process_file(f, OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL, is_global=True)


# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("=" * 60)
    print("Iniciando codificación de variables (transectos + global)")
    print("=" * 60)

    process_by_transect()
    process_global()

    print("\nProceso completado. Revise las carpetas:")
    print(f"  - {OUTPUT_ML_TRANSECT}")
    print(f"  - {OUTPUT_DL_TRANSECT}")
    print(f"  - {OUTPUT_ML_GLOBAL}")
    print(f"  - {OUTPUT_DL_GLOBAL}")

Iniciando codificación de variables (transectos + global)
Procesando: Transecto_1 (global=False)
  ML guardado en /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/encoded/ml/by_transect/Transecto_1.csv
  DL guardado en /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/encoded/dl/by_transect/Transecto_1.csv
Procesando: Transecto_2 (global=False)
  ML guardado en /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/encoded/ml/by_transect/Transecto_2.csv
  DL guardado en /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/encoded/dl/by_transect/Transecto_2.csv
Procesando: T1_E1_Alicante (global=True)
  ML guardado en /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/encoded/ml/global/T1_E1_Alicante.csv
  DL guardado en /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/encoded/dl/global/T1_E1_Alicante.csv
Procesando: T1_E2_Elda (global=True)
  ML guardado en /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/encoded/ml/global/T1_E2_Elda.csv
  DL guardado en /Volumes/copia s

In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import pandas as pd
from pathlib import Path

# ==========================================================
# CONFIGURAR RUTA A INSPECCIONAR
# Cambia por la salida que quieras revisar:
# encoded/ml/by_transect
# encoded/dl/by_transect
# encoded/ml/global
# encoded/dl/global
# ==========================================================

BASE_DIR = "/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/encoded/ml/by_transect"

# ==========================================================
# FUNCIÓN DE DIAGNÓSTICO TEMPORAL
# ==========================================================

def analizar_fechas(csv_file):

    print("\n" + "="*70)
    print(f"Archivo: {csv_file.name}")
    print("="*70)

    df = pd.read_csv(
        csv_file,
        index_col=0,
        parse_dates=True
    )

    # limpiar posibles NaT
    fechas = pd.to_datetime(df.index, errors="coerce")
    fechas = fechas[~pd.isna(fechas)].sort_values()

    if len(fechas) == 0:
        print("No hay fechas válidas.")
        return

    print(f"Primer timestamp : {fechas.min()}")
    print(f"Último timestamp : {fechas.max()}")
    print(f"Número registros : {len(fechas)}")

    # frecuencia inferida
    freq = pd.infer_freq(fechas)

    if freq is not None:
        print(f"Frecuencia inferida: {freq}")
    else:
        print("Frecuencia inferida: irregular o no detectable")

    # diferencias temporales
    diffs = fechas.to_series().diff().value_counts().sort_index()

    print("\nSaltos temporales observados:")
    print(diffs.head(10))

    # si parece horario, comprobar huecos
    try:
        if freq is not None:
            rango_completo = pd.date_range(
                start=fechas.min(),
                end=fechas.max(),
                freq=freq
            )
        else:
            # asumir horario como fallback
            rango_completo = pd.date_range(
                start=fechas.min(),
                end=fechas.max(),
                freq="h"
            )

        faltantes = rango_completo.difference(fechas)

        print(f"\nTimestamps esperados: {len(rango_completo)}")
        print(f"Timestamps faltantes: {len(faltantes)}")

        if len(faltantes) > 0:
            print("\nPrimeros faltantes:")
            print(faltantes[:20])

    except Exception as e:
        print(f"No se pudo evaluar huecos: {e}")

    # resumen por años
    años = pd.Series(fechas.year).value_counts().sort_index()

    print("\nDatos por año:")
    print(años)

    # resumen por meses
    meses = pd.Series(
        fechas.to_period("M")
    ).value_counts().sort_index()

    print("\nPrimeros meses disponibles:")
    print(meses.head(20))


# ==========================================================
# EJECUCIÓN
# ==========================================================

files = sorted(Path(BASE_DIR).glob("*.csv"))

if len(files) == 0:
    print("No se encontraron CSV")
else:
    for f in files:
        analizar_fechas(f)


Archivo: Transecto_1.csv
Primer timestamp : 2024-01-01 00:00:00
Último timestamp : 2025-12-31 00:00:00
Número registros : 17444
Frecuencia inferida: irregular o no detectable

Saltos temporales observados:
datetime
0 days 01:00:00    17380
0 days 02:00:00       55
0 days 03:00:00        6
0 days 04:00:00        1
0 days 08:00:00        1
Name: count, dtype: int64

Timestamps esperados: 17521
Timestamps faltantes: 77

Primeros faltantes:
DatetimeIndex(['2024-05-06 23:00:00', '2024-12-01 20:00:00',
               '2024-12-02 19:00:00', '2024-12-03 18:00:00',
               '2024-12-13 19:00:00', '2024-12-15 19:00:00',
               '2024-12-16 19:00:00', '2024-12-18 19:00:00',
               '2024-12-20 19:00:00', '2024-12-21 19:00:00',
               '2024-12-24 20:00:00', '2024-12-26 21:00:00',
               '2024-12-30 20:00:00', '2024-12-30 21:00:00',
               '2025-01-01 01:00:00', '2025-01-04 20:00:00',
               '2025-01-05 19:00:00', '2025-01-05 23:00:00',
         